### This notebook is uses to draw the PLW Trajectory.

In [ ]:
"""
PLW 15-point trajectory plotting script (front-end config version)
------------------------------------------------------------------
Edit the CONFIG section below to choose mode and GI, then run directly.

Modes:
- MODE = "single": plot one GI file specified by GI (e.g., "-0.25", "0", "2")
- MODE = "all"   : process all CSV files in INPUT_DIR

Notes:
- The script removes consecutive duplicate rows (identical full rows).
- It auto-detects columns named like "x1 deg", "y1 deg", ..., "x15 deg", "y15 deg".
- Output PNGs are saved to OUTPUT_DIR.
- Each marker is a black circle with diameter = 0.75° in data units.
"""

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.collections import PatchCollection
import numpy as np

# -------------------------------------------------------
# CONFIG — edit these values before running
# -------------------------------------------------------
MODE = "all"     # "single" or "all"
GI   = "-0.25"      # Used only when MODE == "single" (e.g., "-0.25", "0", "2")

INPUT_DIR  = Path(r"Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\PLW")
OUTPUT_DIR = Path(r"Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\PLW_pic")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

POINT_DIAMETER_DEG = 0.75   # each point's diameter (in degrees)
STEP = 1                    # sample step; 1 = use all rows
# -------------------------------------------------------


# -------------------------------------------------------
# Utilities
# -------------------------------------------------------
def find_xy_columns(df):
    """
    Find 15 pairs of (x, y) columns based on lowercase prefix matching.
    Accepts suffixes like ' deg' and ignores case.
    """
    cols = [c.strip() for c in df.columns]
    low2orig = {c.lower(): c for c in cols}

    pairs = []
    for i in range(1, 16):
        xprefix = f"x{i}"
        yprefix = f"y{i}"
        xcol = next((low2orig[c] for c in low2orig if c.startswith(xprefix)), None)
        ycol = next((low2orig[c] for c in low2orig if c.startswith(yprefix)), None)
        if xcol is None or ycol is None:
            raise ValueError(
                f"Cannot find both columns for point {i}. "
                f"Expected prefixes '{xprefix}', '{yprefix}'. "
                f"Available: {df.columns.tolist()}"
            )
        pairs.append((xcol, ycol))
    return pairs


def drop_consecutive_duplicates(df):
    """Remove only consecutive duplicate rows (keep the first of each run)."""
    mask_keep = (df != df.shift(1)).any(axis=1)
    mask_keep.iloc[0] = True
    return df.loc[mask_keep].reset_index(drop=True)


def gi_from_filename(path: Path) -> str:
    """Return stem as GI string, e.g., '-0.25.csv' -> '-0.25'."""
    return path.stem


def plot_one_file(csv_path: Path, out_dir: Path):
    """
    Read one CSV file and plot trajectories as black circular dots
    with diameter = 0.75° in data units (not pixels).
    """
    # Try multiple encodings for safety
    for enc in ("utf-8", "gbk", "ansi"):
        try:
            df = pd.read_csv(csv_path, encoding=enc)
            break
        except Exception:
            continue

    xy_pairs = find_xy_columns(df)
    df_xy = df[[c for pair in xy_pairs for c in pair]]
    df_xy = drop_consecutive_duplicates(df_xy).iloc[::STEP].reset_index(drop=True)

    # Create circle patches for all points
    patches = []
    r = POINT_DIAMETER_DEG / 2.0

    for xcol, ycol in xy_pairs:
        xs = pd.to_numeric(df_xy[xcol], errors="coerce")
        ys = pd.to_numeric(df_xy[ycol], errors="coerce")
        for x, y in zip(xs, ys):
            if np.isfinite(x) and np.isfinite(y):
                patches.append(Circle((x, y), r))

    fig, ax = plt.subplots(figsize=(8, 8), dpi=150)
    coll = PatchCollection(patches, facecolor="black", edgecolor="none", alpha=1.0)
    ax.add_collection(coll)

    ax.set_xlabel("Horizontal (deg)")
    ax.set_ylabel("Vertical (deg)")
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, linestyle='--', alpha=0.3)
    ax.set_title(f"PLW Trajectories (GI = {gi_from_filename(csv_path)})")

    # Axis limits with small margins
    all_x = pd.concat([pd.to_numeric(df_xy[x], errors="coerce") for x, _ in xy_pairs])
    all_y = pd.concat([pd.to_numeric(df_xy[y], errors="coerce") for _, y in xy_pairs])
    xmin, xmax = all_x.min() - r, all_x.max() + r
    ymin, ymax = all_y.min() - r, all_y.max() + r
    dx = (xmax - xmin) * 0.05 if xmax > xmin else 1
    dy = (ymax - ymin) * 0.05 if ymax > ymin else 1
    ax.set_xlim(xmin - dx, xmax + dx)
    ax.set_ylim(ymin - dy, ymax + dy)

    # Remove legend (no need for per-point labels)
    # ax.legend().remove()

    out_name = f"PLW_traj_GI_{gi_from_filename(csv_path).replace('-', 'neg').replace('.', 'p')}.png"
    out_path = out_dir / out_name
    fig.tight_layout()
    fig.savefig(out_path)
    plt.show()
    plt.close(fig)
    print(f"[Saved] {out_path}")


# -------------------------------------------------------
# Main flow (driven by CONFIG)
# -------------------------------------------------------
def run_single(gi: str):
    """Plot one GI file."""
    target = INPUT_DIR / f"{gi}.csv"
    if not target.exists():
        candidates = [p for p in INPUT_DIR.glob("*.csv") if gi_from_filename(p) == gi]
        if not candidates:
            raise FileNotFoundError(f"No CSV found for GI = {gi} in {INPUT_DIR}")
        target = candidates[0]
    plot_one_file(target, OUTPUT_DIR)


def run_all():
    """Process all CSV files."""
    files = sorted(INPUT_DIR.glob("*.csv"), key=lambda p: p.stem)
    if not files:
        raise FileNotFoundError(f"No CSV files found in {INPUT_DIR}")
    for p in files:
        plot_one_file(p, OUTPUT_DIR)


if __name__ == "__main__":
    if MODE not in {"single", "all"}:
        raise SystemExit("CONFIG error: MODE must be 'single' or 'all'.")
    if MODE == "single":
        if GI is None or str(GI).strip() == "":
            raise SystemExit("CONFIG error: MODE='single' requires GI string (e.g., '-0.25').")
        run_single(str(GI))
    else:
        run_all()


In [ ]:
# final one

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.collections import PatchCollection

# ============== CONFIG ==============
VALUE_DIR = Path(r"Z:\BioMotionAnlyze\analyze\data\two-dim gauss fit data\exp 202504\heatmap_cor\value map")
PLW_DIR   = Path(r"Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\PLW")
OUT_DIR   = Path(r"Z:\BioMotionAnlyze\analyze\data\two-dim gauss fit data\exp 202504\heatmap_cor\heatmap overlay")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CMAP = "jet"
INTERP = "nearest"

HEATMAP_ALPHA    = 0.9         # 热图透明度
POINT_DIAMETER_DEG = 0.75       # 人形点直径（度）
PLW_ALPHA        = 0.1         # 人形白点透明度
FRAME_DOWNSAMPLE = 4            # 可设 >1 来抽帧（减少“套圈”感）
PAD_DEG          = 0.25       # 轴范围额外留白（度）
VALUE_GLOB       = "GI_*_*.csv" # 兼容 GI_<subject>_<GI>.csv
# ===================================


def load_csv_with_guess(path: Path) -> pd.DataFrame:
    for enc in ("utf-8", "gbk", "ansi"):
        try: return pd.read_csv(path, encoding=enc)
        except Exception: pass
    return pd.read_csv(path)

def parse_gi_from_value_name(path: Path) -> str:
    for token in reversed(path.stem.split("_")):
        try:
            float(token); return token
        except Exception:
            continue
    raise ValueError(f"无法解析 GI: {path.name}")

def load_value_map_df_and_binsize(csv_path: Path):
    df = pd.read_csv(csv_path, index_col=0)
    df.index   = pd.to_numeric(df.index, errors="coerce")
    df.columns = pd.to_numeric(df.columns, errors="coerce")
    df = df.sort_index(axis=0).sort_index(axis=1)

    # 先尝试从 .meta.txt 中读 binsize；失败则由坐标差估计
    binsize = None
    meta = csv_path.with_suffix(".meta.txt")
    if meta.exists():
        try:
            m = re.search(r"grid_bin_size_deg\s*=\s*([0-9.]+)", meta.read_text(encoding="utf-8"))
            if m: binsize = float(m.group(1))
        except Exception:
            pass
    if binsize is None:
        if df.shape[1] > 1:
            binsize = float(np.median(np.diff(df.columns.to_numpy())))
        elif df.shape[0] > 1:
            binsize = float(np.median(np.diff(df.index.to_numpy())))
        else:
            binsize = 0.25
    return df, float(binsize)

def load_plw_csv_for_gi(plw_dir: Path, gi: str) -> pd.DataFrame:
    target = plw_dir / f"{gi}.csv"
    if not target.exists():
        cands = [p for p in plw_dir.glob("*.csv") if p.stem == gi]
        if not cands:
            want = float(gi); best = None
            for p in plw_dir.glob("*.csv"):
                tokens = re.split(r"[_\-\s]+", p.stem) + [p.stem]
                for tk in tokens:
                    try:
                        if abs(float(tk) - want) < 1e-9:
                            best = p; break
                    except Exception:
                        pass
                if best: break
            if best is None:
                raise FileNotFoundError(f"在 {plw_dir} 找不到 GI={gi} 的 PLW CSV")
            target = best
        else:
            target = cands[0]
    return load_csv_with_guess(target)

def find_xy_columns(df: pd.DataFrame, n_points=15):
    cols = [c.strip() for c in df.columns]
    low2orig = {c.lower(): c for c in cols}
    pairs = []
    for i in range(1, n_points+1):
        xcol = next((low2orig[c] for c in low2orig if c.startswith(f"x{i}")), None)
        ycol = next((low2orig[c] for c in low2orig if c.startswith(f"y{i}")), None)
        if xcol is None or ycol is None:
            raise ValueError(f"PLW 缺少 x{i}/y{i} 列。现有列: {df.columns.tolist()}")
        pairs.append((xcol, ycol))
    return pairs

def drop_consecutive_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    mask = (df != df.shift(1)).any(axis=1)
    if len(mask) > 0: mask.iloc[0] = True
    return df.loc[mask].reset_index(drop=True)

def plot_one_overlay(value_csv: Path, plw_df: pd.DataFrame, pairs):
    # --- 读取热图矩阵 + binsize & 计算热图边界（用中心 ± 半格） ---
    df, binsize = load_value_map_df_and_binsize(value_csv)
    x_cent = df.columns.to_numpy(); y_cent = df.index.to_numpy()
    hx_min, hx_max = float(x_cent.min() - binsize/2), float(x_cent.max() + binsize/2)
    hy_min, hy_max = float(y_cent.min() - binsize/2), float(y_cent.max() + binsize/2)

    arr  = df.values.astype(float)
    vmax = float(np.nanmax(arr)) if np.isfinite(np.nanmax(arr)) and np.nanmax(arr) > 0 else 1.0

    # --- 预处理 PLW 点云，计算人形边界 ---
    plw_xy = drop_consecutive_duplicates(plw_df[[c for ab in pairs for c in ab]])
    if FRAME_DOWNSAMPLE > 1:
        plw_xy = plw_xy.iloc[::FRAME_DOWNSAMPLE, :].reset_index(drop=True)

    all_x = []; all_y = []
    for xcol, ycol in pairs:
        all_x.append(pd.to_numeric(plw_xy[xcol], errors="coerce").to_numpy())
        all_y.append(pd.to_numeric(plw_xy[ycol], errors="coerce").to_numpy())
    all_x = np.concatenate(all_x); all_y = np.concatenate(all_y)
    # 去掉 NaN
    all_x = all_x[np.isfinite(all_x)]; all_y = all_y[np.isfinite(all_y)]

    # 人形边界需考虑点半径
    r = POINT_DIAMETER_DEG / 2.0
    if all_x.size > 0 and all_y.size > 0:
        px_min, px_max = float(np.min(all_x) - r), float(np.max(all_x) + r)
        py_min, py_max = float(np.min(all_y) - r), float(np.max(all_y) + r)
    else:
        px_min=px_max=py_min=py_max=0.0

    # --- 最终显示范围 = 热图边界 ∪ 人形边界，再加 PAD ---
    x_min = min(hx_min, px_min) - PAD_DEG
    x_max = max(hx_max, px_max) + PAD_DEG
    y_min = min(hy_min, py_min) - PAD_DEG
    y_max = max(hy_max, py_max) + PAD_DEG
    extent = (hx_min, hx_max, hy_min, hy_max)  # imshow 的范围仍用热图边界

    # --- 绘图 ---
    fig, ax = plt.subplots(figsize=(6.4, 8.4), dpi=150)

    # 1) 先画热图（在下层）
    im = ax.imshow(arr, origin="lower", extent=extent, cmap=CMAP,
                   interpolation=INTERP, vmin=0, vmax=vmax,
                   alpha=HEATMAP_ALPHA, aspect="equal", zorder=1)

    # 2) 再画人形白点（在上层，实心）
    patches = []
    for xcol, ycol in pairs:
        xs = pd.to_numeric(plw_xy[xcol], errors="coerce")
        ys = pd.to_numeric(plw_xy[ycol], errors="coerce")
        for x, y in zip(xs, ys):
            if np.isfinite(x) and np.isfinite(y):
                patches.append(Circle((float(x), float(y)), r))
    coll = PatchCollection(patches, facecolor="white", edgecolor="none",
                           alpha=PLW_ALPHA, zorder=2)
    ax.add_collection(coll)

    # 轴范围用“并集范围”，确保人形完整显示
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    ax.set_xlabel("Horizontal (deg)")
    ax.set_ylabel("Vertical (deg)")
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(value_csv.stem)

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Fixation value (accumulated)")

    fig.tight_layout()
    return fig

def main():
    value_csvs = sorted(VALUE_DIR.glob(VALUE_GLOB))
    if not value_csvs:
        print(f"未找到 {VALUE_GLOB} in {VALUE_DIR}"); return

    gi2files = {}
    for p in value_csvs:
        gi = parse_gi_from_value_name(p)
        gi2files.setdefault(gi, []).append(p)

    print("[Info] GI 分组:", {gi: len(fs) for gi, fs in gi2files.items()})

    for gi, files in gi2files.items():
        plw_df = load_plw_csv_for_gi(PLW_DIR, gi)
        pairs  = find_xy_columns(plw_df)

        for f in files:
            fig = plot_one_overlay(f, plw_df, pairs)
            out = OUT_DIR / f"{f.stem}__overlay.png"
            fig.savefig(out)
            plt.close(fig)
            print(f"[Saved] {out}")

    print("✅ All done! 输出到：", OUT_DIR)

if __name__ == "__main__":
    main()


[Info] GI 分组: {'-0.25': 16, '-0.5': 16, '-1': 16, '-2': 16, '0.25': 16, '0.5': 16, '0': 16, '1': 16, '2': 16}
[Saved] Z:\BioMotionAnlyze\analyze\data\two-dim gauss fit data\exp 202504\heatmap_cor\heatmap overlay\GI_111_-0.25__overlay.png
[Saved] Z:\BioMotionAnlyze\analyze\data\two-dim gauss fit data\exp 202504\heatmap_cor\heatmap overlay\GI_112_-0.25__overlay.png
[Saved] Z:\BioMotionAnlyze\analyze\data\two-dim gauss fit data\exp 202504\heatmap_cor\heatmap overlay\GI_113_-0.25__overlay.png
[Saved] Z:\BioMotionAnlyze\analyze\data\two-dim gauss fit data\exp 202504\heatmap_cor\heatmap overlay\GI_114_-0.25__overlay.png
[Saved] Z:\BioMotionAnlyze\analyze\data\two-dim gauss fit data\exp 202504\heatmap_cor\heatmap overlay\GI_115_-0.25__overlay.png
[Saved] Z:\BioMotionAnlyze\analyze\data\two-dim gauss fit data\exp 202504\heatmap_cor\heatmap overlay\GI_116_-0.25__overlay.png
[Saved] Z:\BioMotionAnlyze\analyze\data\two-dim gauss fit data\exp 202504\heatmap_cor\heatmap overlay\GI_117_-0.25__overla

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Average PLW point coordinates for each Gender Index file and make a summary CSV.
"""

from pathlib import Path
import pandas as pd
import re

# ---------- 路径设置 ----------
INPUT_DIR  = Path(r"Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\PLW")
OUTPUT_DIR = Path(r"Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\PLW_average")

# ---------- 配置 ----------
# 需要的30列
EXPECTED_COLS = [f"{axis}{i}_deg" for i in range(1, 16) for axis in ("x", "y")]
ROUND_DECIMALS = 6

# 文件名解析性别指数模式（-2.csv / 0.25.csv / 2.csv）
GI_PATTERN = re.compile(r"(-?\d+(?:\.\d+)?)\.csv$", re.IGNORECASE)

def parse_gender_index(filename: str):
    m = GI_PATTERN.search(filename)
    if not m:
        return None
    try:
        return float(m.group(1))
    except:
        return None

def average_one(input_csv: Path):
    df = pd.read_csv(input_csv, encoding="utf-8", engine="python")
    cols = [c for c in EXPECTED_COLS if c in df.columns]

    # 转为数值
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    mean_series = df[cols].mean(axis=0).round(ROUND_DECIMALS)
    return {col: mean_series[col] for col in cols}

def main():
    if not INPUT_DIR.exists():
        raise SystemExit(f"输入目录不存在: {INPUT_DIR}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    csv_files = sorted(INPUT_DIR.glob("*.csv"))
    if not csv_files:
        raise SystemExit(f"没找到CSV: {INPUT_DIR}")

    summary_rows = []

    for f in csv_files:
        gi = parse_gender_index(f.name)
        try:
            avg_dict = average_one(f)
        except Exception as e:
            print(f"{f.name} 处理失败: {e}")
            continue

        # 输出单个平均文件
        out_path = OUTPUT_DIR / f.name
        pd.DataFrame([avg_dict]).to_csv(out_path, index=False, encoding="utf-8")
        print(f"✅ {f.name} -> {out_path}")

        row = {"gender_index": gi, "filename": f.name}
        row.update(avg_dict)
        summary_rows.append(row)

    # 构建总汇总表
    def sort_key(r):
        return (1, r["filename"]) if r["gender_index"] is None else (0, r["gender_index"])

    summary_rows.sort(key=sort_key)

    columns = ["gender_index", "filename"] + EXPECTED_COLS
    summary_df = pd.DataFrame(summary_rows, columns=columns)
    summary_csv = OUTPUT_DIR / "PLW_average_summary.csv"
    summary_df.to_csv(summary_csv, index=False, encoding="utf-8")

    print(f"汇总完成 -> {summary_csv}")

if __name__ == "__main__":
    main()


✅ -0.25.csv -> Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\PLW_average\-0.25.csv
✅ -0.5.csv -> Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\PLW_average\-0.5.csv
✅ -1.csv -> Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\PLW_average\-1.csv
✅ -2.csv -> Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\PLW_average\-2.csv
✅ 0.25.csv -> Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\PLW_average\0.25.csv
✅ 0.5.csv -> Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\PLW_average\0.5.csv
✅ 0.csv -> Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\PLW_average\0.csv
✅ 1.csv -> Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\PLW_average\1.csv
✅ 2.csv -> Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\PLW_average\2.csv
🎯 汇总完成 -> Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\PLW_average\PLW_average_summary.csv


In [ ]:

"""
Plot averaged PLW points (only dots, no skeleton lines)
"""

from pathlib import Path
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

# ====== 路径设置 ======
AVG_DIR  = Path(r"Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\PLW_average")
PLOT_DIR = AVG_DIR / "plots_only_points"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# ====== 列名定义 ======
EXPECTED_COLS = [f"{axis}{i}_deg" for i in range(1, 16) for axis in ("x", "y")]

# 文件名解析性别指数
GI_RE = re.compile(r"(-?\d+(?:\.\d+)?)\.csv$", re.IGNORECASE)

def parse_gi(fname):
    m = GI_RE.search(fname)
    return float(m.group(1)) if m else None

def load_points(csv_file: Path):
    df = pd.read_csv(csv_file)
    row = df.iloc[0]
    xs = np.array([row[f"x{i}_deg"] for i in range(1, 16)], dtype=float)
    ys = np.array([row[f"y{i}_deg"] for i in range(1, 16)], dtype=float)

    # 平移到中心，便于对比
    xs = xs - np.mean(xs)
    ys = ys - np.mean(ys)
    return xs, ys

def plot_points(xs, ys, title, save_path):
    plt.figure(figsize=(4,4), dpi=140)
    plt.scatter(xs, ys, s=50, color="black")

    # 标编号（方便定位身体点）
    for i in range(15):
        plt.text(xs[i], ys[i], str(i+1), fontsize=8, ha='left', va='bottom')

    plt.title(title)
    plt.xlabel("x (deg)")
    plt.ylabel("y (deg)")
    plt.gca().set_aspect('equal', adjustable='box')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# ====== 主程序 ======
csv_files = [f for f in AVG_DIR.glob("*.csv") if f.name.lower() != "plw_average_summary.csv"]

if not csv_files:
    raise SystemExit("No average CSV files found!")

# 读取所有点用于统一坐标比例
all_xy = []
for f in csv_files:
    xs, ys = load_points(f)
    all_xy.append((xs, ys))

xs_all = np.concatenate([xy[0] for xy in all_xy])
ys_all = np.concatenate([xy[1] for xy in all_xy])
span = max(xs_all.max()-xs_all.min(), ys_all.max()-ys_all.min()) * 0.7
lims = (-span/2, span/2)

# draw PLW one by one
for f, (xs, ys) in zip(csv_files, all_xy):
    gi = parse_gi(f.name)
    title = f"GI={gi}" if gi is not None else f.name
    save_path = PLOT_DIR / f"{f.stem}.png"
    plt.figure(figsize=(4,4), dpi=140)
    plt.scatter(xs, ys, s=50, color="black")
    for i in range(15):
        plt.text(xs[i], ys[i], str(i+1), fontsize=8, ha='left', va='bottom')
    plt.title(title)
    plt.xlim(lims)
    plt.ylim(lims)
    plt.grid(alpha=0.3)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.tight_layout()
    plt.ylim(-10, 10)
    plt.xlim(-5, 5)
    plt.savefig(save_path)
    plt.close()

print("Done. All point-only plots saved to:", PLOT_DIR)


✅ Done. All point-only plots saved to: Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\PLW_average\plots_only_points
